# Clinical Validation of 3D Mammography AI — Duke BCS-DBT

Standalone validation on **digital breast tomosynthesis (DBT)** — 3D mammography — using the Duke *Breast-Cancer-Screening-DBT* dataset.

Two studies:
1. **Exam-level classification** — a `SliceAggregatorClassifier` lifts a validated 2D model onto the tomosynthesis slice stack (how DBT triage products actually operate). Fully runnable end-to-end.
2. **Lesion localisation (FROC)** — implemented to the *official Duke BCS-DBT* true-positive criterion and 1/2/3/4 FP-per-volume operating points. No DBT detector has public weights, so this notebook scores the FROC engine on a detector's `predictions.csv` (a simulated, clearly-labelled stand-in here — drop in a real one to get real numbers).

> **Heads-up:** the full collection is ~1.5 TB. Work from a *subset* — the cells below pull a small number of validation-split volumes.

## 1 · Environment setup
Runs on **Colab** or **Kaggle**. On Kaggle, set *Accelerator* to a **GPU** and turn *Internet* **On** (required for the installs, the TCIA download, the model, and cloning the project code). The setup cell below auto-clones the `mammoval` package if needed.

In [ ]:
!pip install -q transformers timm pydicom pylibjpeg pylibjpeg-libjpeg pylibjpeg-openjpeg tcia_utils
# torch / torchvision are pre-installed on Colab and Kaggle.

In [ ]:
# Make the `mammoval` package importable. Auto-clones the project repo if it
# is not already present (Internet must be ON for the clone on Kaggle).
import os, sys, glob, subprocess

REPO_URL = 'https://github.com/Joana-Mansa/breast-ai-clinical-validation.git'

def _find_mammoval():
    cands = ['.', 'breast-ai-clinical-validation',
             '/content/breast-ai-clinical-validation']
    cands += glob.glob('/kaggle/input/*') + glob.glob('/kaggle/input/*/*')
    cands += glob.glob('/kaggle/working/*')
    for c in cands:
        if c and os.path.isdir(os.path.join(c, 'mammoval')):
            return os.path.abspath(c)
    return None

loc = _find_mammoval()
if loc is None:
    print('cloning', REPO_URL, '...')
    subprocess.run(['git', 'clone', '-q', REPO_URL], check=False)
    loc = _find_mammoval()
if loc is None:
    raise RuntimeError('mammoval not found and clone failed - turn Internet ON, '
                       'or add the project folder as a Kaggle dataset (+ Add Input).')
sys.path.insert(0, loc)
import mammoval
print('mammoval', mammoval.__version__, 'ready (from ' + loc + ')')

## 2 · Download a subset of Duke BCS-DBT
The dataset lives on TCIA: collection **`Breast-Cancer-Screening-DBT`** (DOI 10.7937/E4WT-CD02). You need two things:

1. **The three metadata CSVs** — `BCS-DBT-file-paths-*.csv`, `BCS-DBT-labels-*.csv`, `BCS-DBT-boxes-*.csv` — from the TCIA collection page (small files). Place them under `data/duke-dbt/`.
2. **A subset of DICOM volumes.** The cell below uses `tcia_utils` to fetch a handful of validation-split series. Downloading all of it is neither necessary nor practical in Colab.

Edit `N_SERIES` to control how much you pull.

In [ ]:
import os, glob, shutil
ON_KAGGLE = os.path.exists('/kaggle/input')
os.makedirs('data/duke-dbt', exist_ok=True)

# --- DICOM subset via the TCIA API (needs Internet ON) ---------------
from tcia_utils import nbia
N_SERIES = 40        # number of DBT volumes to pull (keep small)
series = nbia.getSeries(collection='Breast-Cancer-Screening-DBT')
nbia.downloadSeries(series[:N_SERIES], path='data/duke-dbt')
print('downloaded', N_SERIES, 'series')

# --- metadata CSVs: BCS-DBT-{file-paths,labels,boxes}-*.csv ----------
# Get these small files from the TCIA collection page.
if ON_KAGGLE:
    # Add the CSVs as a Kaggle dataset (+ Add Input), then this copies
    # them next to the DICOMs so the loader sees one root:
    for src in glob.glob('/kaggle/input/*/BCS-DBT-*.csv'):
        shutil.copy(src, 'data/duke-dbt')
    print('metadata CSVs:', sorted(os.listdir('data/duke-dbt')))
else:
    from google.colab import files
    print('Upload the three BCS-DBT-*.csv metadata files:')
    for name in files.upload():
        os.replace(name, os.path.join('data/duke-dbt', name))

## 3 · Load the dataset
`DukeDBTDataset` locates the CSVs by glob (filenames vary across TCIA snapshots) and keeps only the views whose DICOM is actually on disk — the right behaviour for a partial download.

In [ ]:
from mammoval.data import DukeDBTDataset
import json

ds = DukeDBTDataset(root='data/duke-dbt', split='validation',
                    require_local=True)
print(json.dumps(ds.summary(), indent=2, default=str))
ds.cases.head()

## 4 · Build the 3D classifier
A DBT exam is a stack of reconstructed slices. `SliceAggregatorClassifier` scores a sample of slices with a 2D classifier and pools them (top-k mean).

For the 2D base we use the same **linear probe** as the 2D notebook, fit on CBIS-DDSM. On Kaggle, also **+ Add Input** the *"CBIS-DDSM Breast Cancer Image Dataset"* (awsaf49) — the cell auto-detects it. Applying a 2D mammography classifier to tomosynthesis slices is a reasonable, runnable baseline; a native 3D detector would be stronger (see the FROC notes below).

In [ ]:
import os, glob
from mammoval.data import CBISDDSMDataset
from mammoval.models import LinearProbeClassifier, SliceAggregatorClassifier

# locate the CBIS-DDSM mount (added as a second Kaggle input)
cbis = [d for d in sorted(glob.glob('/kaggle/input/*'))
        if os.path.isdir(os.path.join(d, 'csv'))
        and os.path.isdir(os.path.join(d, 'jpeg'))]
assert cbis, 'Add the awsaf49 CBIS-DDSM image dataset as a 2nd input.'
cbis_train = CBISDDSMDataset(root=cbis[0], split='train')

base_2d = LinearProbeClassifier(backbone='resnet50')
base_2d.fit(cbis_train, limit=800)      # fit the 2D head on mammograms
model = SliceAggregatorClassifier(base_2d, n_slices=12,
                                  aggregate='top_k_mean', top_k=3)
print('volume model:', model.name)

## 5 · Run inference → predictions table
DBT volumes are large; keep `LIMIT` modest in Colab.

In [ ]:
from mammoval.models import score_dataset

LIMIT = 40
preds = score_dataset(model, ds, limit=LIMIT).dropna(subset=['y_score'])
preds[['case_id','y_true','y_score','view','label_class']].head()

## 6 · Exam-level classification validation

In [ ]:
from mammoval.pipeline import ClassificationConfig, run_classification_validation

cfg = ClassificationConfig(
    subgroup_cols=('view', 'laterality'),
    target_specificities=(0.90, 0.96),
    is_probability=True,
    dataset_name='Duke BCS-DBT (validation subset)',
)
results = run_classification_validation(preds, cfg)
d = results['discrimination']
print('AUC = %.3f   95%% CI %.3f-%.3f' % (d['auc'], *d['auc_ci']))

## 7 · Lesion localisation — FROC (official Duke metric)
FROC scores each *mark*: a mark is a true positive only if its centre is within `max(√(W²+H²)/2, 100)` px of a ground-truth lesion **and** within `VolumeSlices/4` slices of it — the exact `duke_dbt_hit` criterion. The mean sensitivity at 1/2/3/4 FP/volume is the official DBTex ranking metric.

A real DBT detector emits `predictions.csv` with columns `case_id, score, x, y, slice`. No such model has public weights, so the cell below **simulates** that file from the ground-truth boxes — clearly an illustrative stand-in. Replace `predictions` with a real detector's output to get real localisation numbers.

In [ ]:
import numpy as np, pandas as pd

# ---- ILLUSTRATIVE simulated detector output (replace with real CSV) --
rng = np.random.default_rng(0)
rows = []
for cid in ds.cases['case_id']:
    for les in ds.lesion_boxes(cid):          # GT lesions -> noisy hits
        if rng.random() < 0.7:
            rows.append(dict(case_id=cid, score=float(rng.beta(5,2)),
                             x=les['x']+rng.normal(0,40),
                             y=les['y']+rng.normal(0,40),
                             slice=les['slice']+rng.normal(0,4)))
    for _ in range(rng.poisson(1.2)):          # false marks
        rows.append(dict(case_id=cid, score=float(rng.beta(2,5)),
                         x=rng.uniform(0,2000), y=rng.uniform(0,2000),
                         slice=rng.uniform(0,60)))
predictions = pd.DataFrame(rows)
# ----------------------------------------------------------------------

from mammoval.metrics import duke_dbt_hit
from mammoval.pipeline import run_localization_validation

froc_cases = ds.assemble_froc_cases(predictions)
loc = run_localization_validation(froc_cases, duke_dbt_hit,
                                  fp_points=(1,2,3,4),
                                  dataset_name='Duke BCS-DBT (localisation)')
print('FROC mean sensitivity = %.3f  (official Duke metric)'
      % loc['mean_sensitivity'])

## 8 · Generate the HTML validation report

In [ ]:
from mammoval.report import build_report
from IPython.display import HTML

os.makedirs('outputs', exist_ok=True)
path = build_report(
    results, localization_results=loc,
    output_path='outputs/duke_dbt_validation_report.html',
    title='3D Mammography AI — Clinical Validation (Duke BCS-DBT)',
    extra_limitations=[
        'Exam scores come from a 2D model aggregated over slices, not '
        'a native 3D model — a deliberate, documented stand-in.',
        'FROC here is computed on a SIMULATED detector output; the '
        'curve is illustrative until a real predictions.csv is supplied.',
        'Only a small validation subset was downloaded — widen it for '
        'a stable estimate.',
    ])
HTML(open(path, encoding='utf-8').read())

## Notes & next steps
* **Real localisation numbers** need a trained DBT detector (`mateuszbuda/duke-dbt-detection` or `mazurowski-lab/DBTex-baseline` provide training code). Export its marks as `predictions.csv` and re-run section 7 — nothing else changes.
* **DICOM fidelity.** `DukeDBTDataset.load` uses a plain pydicom read. For localisation that must align with the ground-truth boxes, use the official `dcmread_image` (laterality correction) from `mazurowski-lab/duke-dbt-data`.
* The FROC engine and `duke_dbt_hit` criterion are unit-tested against the published metric definition — see `tests/test_localization.py`.